The below demonstrates ways to call a tool using LLM - Set ticket prices 

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

#globalise the ticket price to have set function persist over the kernel session
current_prices= {"london": 799, "delhi":200, "paris": 1000, "panipat":100}

# The tool function meta data that acts as a bridge for LLM and application functions
# The get tool function gets ticket prices from dictionary
get_price_function = {
        "name": "get_ticket_price",
        "description": "Get the price of a return ticket to the destination city.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string",
                    "description": "The city that the customer wants to travel to",
                },
            },
            "required": ["destination_city"],
            "additionalProperties": False
        }
    }
    
# The set tool function sets ticket prices in dictionary
set_price_function = {
    "name": "set_ticket_price",
    "description": "Sets the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer told to update the price of",
            },
            "price_being_set": {
                "type": "number",
                "description": "The price being set for the destination city"
            }
        },
        "required": ["destination_city", "price_being_set"],
        "additionalProperties": False
    }
}
# Define a JSON array list to define the tools type
tools= [
            {"type":"function", "function": get_price_function}, 
            {"type":"function", "function": set_price_function}
        ]

# Gets the ticket price from dictionary
def get_ticket_price(destination_city):
    current_price=current_prices.get(destination_city.lower(), -1)
    if current_price==-1:
        return f"No price data available for {destination_city}"
    else:
        return f"The price of a ticket to {destination_city} is ${current_price}"

# Sets the ticket price in dictionary
def set_ticket_price(destination_city, price_being_set):
    current_prices[destination_city.lower()]=price_being_set
    response_content= (
        f"The ticket price for {destination_city} has been updated "
        f"to ${price_being_set}"
    ) 
    return response_content

#add the function dictionary for a strategy like pattern
available_functions= {"get_ticket_price": get_ticket_price, "set_ticket_price":set_ticket_price}


#initalise the load env and openai
load_dotenv(override=True)
openai_api_key= os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI api key is good")
else:
    print("OpenAI api key is not set")
    
#set the model
MODEL = "gpt-4.1-mini"
openai=OpenAI(api_key=openai_api_key)
#set the system prompt
system_message="""you are a helpful assistant for Airlines company called shubham airlines. You help in giving ticket prices various destination. If you do not know the answer, say so. """


# The generic tool calling handler to call different function decided on function_name
def handle_tool_call(message):
    tool_call=message.tool_calls[0]
    tool_name=tool_call.function.name
    tool_args=json.loads(tool_call.function.arguments)
    if tool_name in available_functions:
        function_for_tool=available_functions[tool_name]
        response_content=function_for_tool(**tool_args)
        response = {
            "role":"tool",
            "content": response_content,
            "tool_call_id": tool_call.id
        }
        return response
    else:
        raise ValueError(
                f"No available tool found for {tool_name}"
            ) 
    
    
# Generic chat function -> The history is managed by gradio and hence it's simpler.
def chat(message, history):
    # a standard practice is to conver history into list of dictoneries and append to user prompt
    history=[{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{"role":"system", "content":system_message}] + history + [{"role": "user", "content":message}]
    
    # call the openai api
    response= openai.chat.completions.create(model=MODEL, messages=messages, stream=False, tools=tools)
    # if it ends with tool calls then call the tool explicitly -> it's upto LLM to decide when to call llm
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content


# Invoke gradio chat function with call back as chat function
gr.ChatInterface(fn=chat, type="messages").launch()

    